# Functional Data Structures

Collections are one of the most used data structures in functional programming, and where we can immediately see the advantages of using it.

In Scala, the sequence Seq is one of the most common data structures, representing an ordered sequence of elements. It can be mutable or immutable.

The Seq `apply` method will create by default a concrete implementation of a sequence, which is a `List`.


In [1]:
Seq.apply(34,56,78)

res0: Seq[Int] = List(34, 56, 78)

As usual the apply method is implied by using `()`:

In [ ]:
val seq1 = Seq(4,8,7)

Sequences can also be built using the prepend methods. Notice the `Nil` element which is in fact the empty sequence.

In [2]:
val seq2 = 3 +: 4 +: 5 +: Nil

seq2: List[Int] = List(3, 4, 5)

In [3]:
seq2.head

res2: Int = 3

In [4]:
seq2.tail

res3: List[Int] = List(4, 5)

Instead of lists, in which we typically access the head, and then the "next" element in the list, we can also use a `Vector`, where direct access to each element is possible through the index. 

Construction and other operations follow a ver similar pattern, but the internal implementation is different.

In [5]:
val vec1 = Vector(3,5,6)

val vec2 = 3 +: 5 +: 6 +: Vector.empty

vec1: Vector[Int] = Vector(3, 5, 6)
vec2: Vector[Int] = Vector(3, 5, 6)

Sets imply no duplicates in the collection. One implementation of it is a `HashSet`

In [6]:
val set1 = Set(4,6,7,4,3,5,6)

set1: Set[Int] = HashSet(5, 6, 7, 3, 4)

### Iterable

The iterable trait makes it possible to iterate over the element of a collection. Most of the colleciton classes will implement it. 

This allows to use methods such as `foreach`. We can implement our own iterable class.

In [12]:
class MyList[A](data:(A,A,A)) extends IterableOnce[A]:
  def foreach[U](f: A => U):Unit =
    for e <- this.data.toList do
      f(e)
 
  def iterator:Iterator[A] = ???

defined class MyList

Notice that the `foreach` return is always `Unit`, regardless of what function we pass as parameter. 

We can say it is not very "pure".

In [13]:
val mymy = new MyList[Char](('o','l','a'))
mymy.foreach(e=>println(e))

o
l
a


mymy: MyList[Char] = ammonite.$sess.cmd11$Helper$MyList@6776addc

### Foreach

Looking more into `foreach`, let's see its signature:

```
  def foreach[U](f: A => U): Unit
```

One function we can pass and that is commonly used is a `match` `case`:

## 

In [14]:
val seq = Seq(4,"2",7,"3",2)

seq.foreach(x => x match {
  case s:String => println(s"string $s")
  case i:Int => println(s"int $i")
})

int 4
string 2
int 7
string 3
int 2


seq: Seq[Matchable] = List(4, "2", 7, "3", 2)

In [17]:
seq.foreach {
  case s:String => println(s"string $s")
  case i:Int => println(s"int $i")
}

int 4
string 2
int 7
string 3
int 2


### map

Looking into the signature of map, we can see that essentially there is a type transformation form `A` to `B`

```
class C[A]:
  def map[B](f: (A) => B): C[B]
```

In [20]:
case class Person(name:String, age:Int)

val persons = Seq(Person("amy",4),Person("jim",6),Person("tim",4))

defined class Person
persons: Seq[Person] = List(
  Person(name = "amy", age = 4),
  Person(name = "jim", age = 6),
  Person(name = "tim", age = 4)
)

In [22]:
persons.map {
  case Person(name,4) => s"$name is in 1st Grade"
  case Person(name,_) => s"$name is in Another Grade"
}

res21: Seq[String] = List(
  "amy is in 1st Grade",
  "jim is in Another Grade",
  "tim is in 1st Grade"
)

In [24]:
val grades = (1 to 3).map(e => Seq(5,6,4))

grades: IndexedSeq[Seq[Int]] = Vector(
  List(5, 6, 4),
  List(5, 6, 4),
  List(5, 6, 4)
)

### flatMap

Looking at the signature of flatmap, it looks similar to map, only that the return of the function parameter is often a collection `C`, so the parameter type of the class. 

```
class C[A]:
  def flatMap[B](f: A => Seq[B]): C[B]
  def map[B](f: (A) => B): C[B]
```

In [25]:
(1 to 5).flatMap(i => (1 to i))

res24: IndexedSeq[Int] = Vector(1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5)

Flatmap can be very useful when simplifying outputs of other compound types, like when we have collection of `Option`:

In [27]:
val grades = Seq(3,5,2,6,5,2)

def grading(grade:Int):Option[Int] = if grade >= 4 then Some(grade) else None

grades.map(e=>grading(e))

grades.flatMap(e=>grading(e))

grades: Seq[Int] = List(3, 5, 2, 6, 5, 2)
defined function grading
res26_2: Seq[Option[Int]] = List(
  None,
  Some(value = 5),
  None,
  Some(value = 6),
  Some(value = 5),
  None
)
res26_3: Seq[Int] = List(5, 6, 5)

### fold and reduce and all the rest

Looking at the signature of reduce, we see that all types are esentially the same, all the pairwise elements to operate on, as well as the return.

```
def reduce[A1 >: A](op: (A1, A1) => A1): A1
```

In [28]:
grades.reduce((x,y) => x+y)

res27: Int = 23

In `foldLeft` the signature is very similar, but it introduces a "first element" `z`, and therefore the return type can actually change.

```
def foldLeft[B](z: B)(op: (B, A) => B): B
```

In [29]:
grades.foldLeft(0)((x,y) => x+y)

res28: Int = 23

In [34]:
grades.foldLeft(0)((x:Int,y:Int) => x+y)

res33: Int = 23

In [30]:
grades.foldLeft(1)((x,y) => x+y)

res29: Int = 24

So with `fold` in general we have further freedom to modify the types in different ways.

In [31]:
grades.foldLeft("The grades are: ")((x,y) => x+y)

res30: String = "The grades are: 352652"

In [33]:
grades.foldLeft("The grades are: ")((x,y) => s"$x $y")

res32: String = "The grades are:  3 5 2 6 5 2"

In [40]:
grades.foldRight(Seq.empty){(x,y) => (x+0.5) +: y}

res39: Seq[Any] = List(3.5, 5.5, 2.5, 6.5, 5.5, 2.5)